# Markov Chain Analysis of Sequential Behaviour in AI Responses

Encoding sentence-level behavioural states within each response and estimating
transition matrices per experimental condition to test whether adversarial
pressure produces **drift toward unsafe behavioural states**.

**Approach:** Each response is segmented into sentences. Each sentence is
classified into a behavioural state based on its linguistic properties.
Transition matrices are estimated per condition, and we test whether
high-pressure conditions shift the stationary distribution toward unsafe states.

**Contents:**
1. Sentence segmentation & behavioural state classification
2. Transition matrix estimation (overall & per condition)
3. Stationary distribution analysis
4. Pressure-conditional drift testing
5. Absorbing-state analysis
6. Log findings to `results/markov/metrics.json`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from datetime import datetime
from scipy import stats as sp_stats
from collections import Counter

sns.set_theme(style='whitegrid', palette='colorblind', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

SEED = 42
np.random.seed(SEED)
FIGURES_DIR = Path('../reports/figures')
RESULTS_DIR = Path('../results')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')

In [ ]:
# Load data
d2a = pd.read_parquet('../data/processed/d2a_sessions.parquet')
d2b = pd.read_parquet('../data/processed/d2b_turns.parquet')
d2c = pd.read_parquet('../data/processed/d2c_labels.parquet')

# Merge session metadata with text and labels
df = d2a.merge(d2b[['session_id', 'content']], on='session_id')
df = df.merge(d2c[['session_id', 'overall_unsafe']], on='session_id', suffixes=('', '_label'))

# Clean up merge artefacts
unsafe_col = next(c for c in df.columns if c.startswith('overall_unsafe'))
if unsafe_col != 'overall_unsafe':
    df.rename(columns={unsafe_col: 'overall_unsafe'}, inplace=True)
    df.drop(columns=[c for c in df.columns if 'overall_unsafe' in c and c != 'overall_unsafe'],
            errors='ignore', inplace=True)

y = df['overall_unsafe'].astype(int).values

print(f'Sessions: {len(df)}')
print(f'Unsafe: {y.sum()} ({y.mean():.1%})')

## 1. Sentence Segmentation & Behavioural State Classification

Each sentence is classified into one of 5 behavioural states:
- **citing** — contains NCT IDs, references, or specific evidence citations
- **hedging** — epistemic uncertainty (may, might, possibly, suggests, unclear)
- **claiming** — confident assertions without hedging or citation
- **refusing** — explicit refusal or safety-related pushback
- **neutral** — structural/transitional content (headings, lists, connectors)

In [ ]:
# Sentence segmentation
def segment_sentences(text):
    """Split response text into sentences, filtering very short fragments."""
    # Split on sentence boundaries (period, exclamation, question, newline)
    sentences = re.split(r'(?<=[.!?])\s+|\n+', str(text))
    # Filter out very short fragments (< 15 chars) and empty strings
    sentences = [s.strip() for s in sentences if len(s.strip()) >= 15]
    return sentences

# Behavioural state classifier
HEDGING_WORDS = [
    'may', 'might', 'could', 'possibly', 'perhaps', 'suggests',
    'uncertain', 'unclear', 'likely', 'appears', 'seems', 'potential',
    'approximately', 'estimated', 'limited evidence', 'inconclusive',
    'not definitive', 'warrants further', 'caution', 'preliminary'
]

CERTAINTY_WORDS = [
    'clearly', 'definitely', 'certainly', 'undoubtedly', 'proven',
    'conclusively', 'demonstrates', 'establishes', 'confirms',
    'without doubt', 'unequivocally'
]

REFUSAL_WORDS = [
    'cannot', 'should not', 'i cannot', 'i\'m unable', 'not appropriate',
    'beyond my', 'refuse', 'inappropriate', 'ethical concern',
    'not recommend', 'strongly advise against'
]

NCT_PATTERN = re.compile(r'NCT\d{5,}', re.IGNORECASE)

def classify_sentence(sentence):
    """Classify a sentence into a behavioural state."""
    s_lower = sentence.lower()
    
    # Priority 1: Refusing (safety-critical)
    if any(w in s_lower for w in REFUSAL_WORDS):
        return 'refusing'
    
    # Priority 2: Citing (evidence-grounded)
    if NCT_PATTERN.search(sentence) or 'et al' in s_lower or 'doi:' in s_lower:
        return 'citing'
    
    # Priority 3: Hedging (uncertainty)
    hedge_count = sum(1 for w in HEDGING_WORDS if w in s_lower)
    certainty_count = sum(1 for w in CERTAINTY_WORDS if w in s_lower)
    
    if hedge_count >= 2 or (hedge_count >= 1 and certainty_count == 0):
        return 'hedging'
    
    # Priority 4: Claiming (confident assertions)
    if certainty_count >= 1 or len(sentence) > 80:
        return 'claiming'
    
    # Default: Neutral
    return 'neutral'

# Apply to all sessions
STATE_NAMES = ['citing', 'hedging', 'claiming', 'refusing', 'neutral']
state_to_idx = {s: i for i, s in enumerate(STATE_NAMES)}
n_states = len(STATE_NAMES)

all_sequences = []
for idx, row in df.iterrows():
    sentences = segment_sentences(row['content'])
    states = [classify_sentence(s) for s in sentences]
    all_sequences.append(states)

df['state_sequence'] = all_sequences
df['n_sentences'] = [len(seq) for seq in all_sequences]

# Summary
all_states_flat = [s for seq in all_sequences for s in seq]
state_counts = Counter(all_states_flat)
total = len(all_states_flat)

print(f'Total sentences: {total}')
print(f'Sentences per response: {df["n_sentences"].describe().round(1).to_string()}')
print(f'\nState distribution:')
for state in STATE_NAMES:
    print(f'  {state:<12} {state_counts[state]:>5} ({state_counts[state]/total:.1%})')

## 2. Transition Matrix Estimation

For each pair of consecutive sentences $(s_t, s_{t+1})$, count the transitions
between behavioural states. Normalise to obtain the transition probability matrix $T$
where $T_{ij} = P(s_{t+1} = j \mid s_t = i)$.

In [ ]:
def estimate_transition_matrix(sequences, state_names, laplace_smooth=1):
    """Estimate transition matrix from a list of state sequences."""
    n = len(state_names)
    state_idx = {s: i for i, s in enumerate(state_names)}
    counts = np.zeros((n, n)) + laplace_smooth  # Laplace smoothing
    
    for seq in sequences:
        for t in range(len(seq) - 1):
            i = state_idx[seq[t]]
            j = state_idx[seq[t + 1]]
            counts[i, j] += 1
    
    # Normalise rows to get probabilities
    row_sums = counts.sum(axis=1, keepdims=True)
    T = counts / row_sums
    return T, counts - laplace_smooth  # Return raw counts too

# Overall transition matrix
T_overall, raw_counts = estimate_transition_matrix(all_sequences, STATE_NAMES)

print('=== Overall Transition Matrix ===')
T_df = pd.DataFrame(T_overall, index=STATE_NAMES, columns=STATE_NAMES).round(3)
print(T_df.to_string())

print(f'\nRaw transition counts:')
print(pd.DataFrame(raw_counts.astype(int), index=STATE_NAMES, columns=STATE_NAMES).to_string())

In [ ]:
# Visualise the overall transition matrix as a heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: transition probability heatmap
ax = axes[0]
sns.heatmap(T_df, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
            vmin=0, vmax=0.6, linewidths=0.5, square=True)
ax.set_title('Transition Probabilities P(next | current)')
ax.set_xlabel('Next State')
ax.set_ylabel('Current State')

# Right: state frequency bar chart
ax = axes[1]
state_props = [state_counts[s] / total for s in STATE_NAMES]
colors = ['#3498db', '#f39c12', '#e74c3c', '#27ae60', '#95a5a6']
ax.barh(STATE_NAMES, state_props, color=colors, alpha=0.8)
ax.set_xlabel('Proportion of Sentences')
ax.set_title('Behavioural State Distribution')
for i, v in enumerate(state_props):
    ax.text(v + 0.01, i, f'{v:.1%}', va='center', fontsize=9)

plt.suptitle('Sentence-Level Behavioural State Transitions', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '31_markov_transition_matrix.png', bbox_inches='tight')
plt.show()

## 3. Stationary Distribution Analysis

The stationary distribution $\pi$ of the Markov chain represents the long-run
proportion of time spent in each state. If pressure conditions shift $\pi$
toward "claiming" and away from "hedging" or "citing", this indicates
systematic behavioural drift under adversarial conditions.

In [ ]:
def stationary_distribution(T):
    """Compute the stationary distribution of a transition matrix."""
    eigenvalues, eigenvectors = np.linalg.eig(T.T)
    # Find eigenvector for eigenvalue closest to 1
    idx = np.argmin(np.abs(eigenvalues - 1.0))
    pi = np.real(eigenvectors[:, idx])
    pi = pi / pi.sum()  # Normalise
    return pi

# Compute per-condition transition matrices and stationary distributions
conditions = {
    'pressure_id': sorted(df.pressure_id.unique()),
    'scenario_id': sorted(df.scenario_id.unique()),
    'monitoring_id': sorted(df.monitoring_id.unique()),
    'overall_unsafe': [False, True]
}

# Store results
condition_results = {}

for cond_col, cond_values in conditions.items():
    condition_results[cond_col] = {}
    print(f'\n=== {cond_col} ===')
    print(f'{"Condition":<28}', '  '.join(f'{s:<10}' for s in STATE_NAMES))
    print('-' * 85)
    
    for val in cond_values:
        mask = df[cond_col] == val
        seqs = [all_sequences[i] for i in range(len(df)) if mask.iloc[i]]
        
        T_cond, _ = estimate_transition_matrix(seqs, STATE_NAMES)
        pi = stationary_distribution(T_cond)
        
        condition_results[cond_col][str(val)] = {
            'transition_matrix': T_cond.tolist(),
            'stationary_dist': pi.tolist(),
            'n_sessions': int(mask.sum()),
            'n_transitions': sum(len(s) - 1 for s in seqs)
        }
        
        print(f'{str(val):<28}', '  '.join(f'{p:<10.3f}' for p in pi))

In [ ]:
# Visualise stationary distributions across conditions
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for ax, (cond_col, cond_values) in zip(axes.ravel(), conditions.items()):
    pi_data = []
    labels = []
    
    for val in cond_values:
        pi = condition_results[cond_col][str(val)]['stationary_dist']
        pi_data.append(pi)
        labels.append(str(val))
    
    pi_array = np.array(pi_data)
    x = np.arange(n_states)
    width = 0.8 / len(labels)
    
    for i, (label, pi) in enumerate(zip(labels, pi_data)):
        offset = (i - len(labels)/2 + 0.5) * width
        ax.bar(x + offset, pi, width, label=label, alpha=0.8)
    
    ax.set_xticks(x)
    ax.set_xticklabels(STATE_NAMES, fontsize=9)
    ax.set_ylabel('Stationary Probability')
    title = cond_col.replace('_id', '').replace('_', ' ').title()
    ax.set_title(f'Stationary Distribution by {title}')
    ax.legend(fontsize=7, ncol=2)
    ax.set_ylim(0, 0.6)

plt.suptitle('Markov Chain Stationary Distributions by Condition', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '32_markov_stationary_distributions.png', bbox_inches='tight')
plt.show()

## 4. Pressure-Conditional Drift Testing

Formally test whether pressure conditions shift the transition dynamics
using a chi-squared test of homogeneity on the transition count matrices.

$H_0$: The transition matrix is the same across all pressure conditions.
$H_1$: At least one pressure condition has a different transition structure.

In [ ]:
# Chi-squared test of transition homogeneity: pressure conditions
print('=== Chi-Squared Test of Transition Homogeneity ===')
print('Testing whether pressure conditions share the same transition dynamics.\n')

for from_state_idx, from_state in enumerate(STATE_NAMES):
    # Build contingency table: rows = pressure conditions, columns = next state
    contingency = []
    pressure_labels = []
    
    for pressure in sorted(df.pressure_id.unique()):
        mask = df.pressure_id == pressure
        seqs = [all_sequences[i] for i in range(len(df)) if mask.iloc[i]]
        
        # Count transitions FROM this state
        row = np.zeros(n_states)
        for seq in seqs:
            for t in range(len(seq) - 1):
                if seq[t] == from_state:
                    row[state_to_idx[seq[t + 1]]] += 1
        
        if row.sum() > 0:
            contingency.append(row)
            pressure_labels.append(pressure)
    
    contingency = np.array(contingency)
    
    # Remove columns with all zeros
    nonzero_cols = contingency.sum(axis=0) > 0
    if nonzero_cols.sum() < 2 or len(contingency) < 2:
        print(f'  From {from_state:<12}: insufficient data for test')
        continue
    
    contingency_filtered = contingency[:, nonzero_cols]
    
    chi2, p_val, dof, expected = sp_stats.chi2_contingency(contingency_filtered)
    sig = '*' if p_val < 0.05 else ''
    print(f'  From {from_state:<12}: chi2={chi2:>7.2f}, p={p_val:.4f}, dof={dof} {sig}')

# Same test for scenario conditions
print('\n=== Chi-Squared Test: Scenario Conditions ===')

for from_state_idx, from_state in enumerate(STATE_NAMES):
    contingency = []
    
    for scenario in sorted(df.scenario_id.unique()):
        mask = df.scenario_id == scenario
        seqs = [all_sequences[i] for i in range(len(df)) if mask.iloc[i]]
        
        row = np.zeros(n_states)
        for seq in seqs:
            for t in range(len(seq) - 1):
                if seq[t] == from_state:
                    row[state_to_idx[seq[t + 1]]] += 1
        
        if row.sum() > 0:
            contingency.append(row)
    
    contingency = np.array(contingency)
    nonzero_cols = contingency.sum(axis=0) > 0
    if nonzero_cols.sum() < 2 or len(contingency) < 2:
        print(f'  From {from_state:<12}: insufficient data for test')
        continue
    
    contingency_filtered = contingency[:, nonzero_cols]
    chi2, p_val, dof, expected = sp_stats.chi2_contingency(contingency_filtered)
    sig = '*' if p_val < 0.05 else ''
    print(f'  From {from_state:<12}: chi2={chi2:>7.2f}, p={p_val:.4f}, dof={dof} {sig}')

In [ ]:
# Visualise transition matrices: safe vs unsafe sessions
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Safe sessions
safe_seqs = [all_sequences[i] for i in range(len(df)) if y[i] == 0]
unsafe_seqs = [all_sequences[i] for i in range(len(df)) if y[i] == 1]

T_safe, _ = estimate_transition_matrix(safe_seqs, STATE_NAMES)
T_unsafe, _ = estimate_transition_matrix(unsafe_seqs, STATE_NAMES)

T_safe_df = pd.DataFrame(T_safe, index=STATE_NAMES, columns=STATE_NAMES)
T_unsafe_df = pd.DataFrame(T_unsafe, index=STATE_NAMES, columns=STATE_NAMES)

# Difference matrix
T_diff = T_unsafe - T_safe
T_diff_df = pd.DataFrame(T_diff, index=STATE_NAMES, columns=STATE_NAMES)

sns.heatmap(T_safe_df, annot=True, fmt='.2f', cmap='Blues', ax=axes[0],
            vmin=0, vmax=0.6, linewidths=0.5, square=True)
axes[0].set_title(f'Safe Sessions (n={len(safe_seqs)})')
axes[0].set_xlabel('Next'); axes[0].set_ylabel('Current')

sns.heatmap(T_unsafe_df, annot=True, fmt='.2f', cmap='Reds', ax=axes[1],
            vmin=0, vmax=0.6, linewidths=0.5, square=True)
axes[1].set_title(f'Unsafe Sessions (n={len(unsafe_seqs)})')
axes[1].set_xlabel('Next'); axes[1].set_ylabel('Current')

sns.heatmap(T_diff_df, annot=True, fmt='+.2f', cmap='RdBu_r', ax=axes[2],
            vmin=-0.2, vmax=0.2, linewidths=0.5, center=0, square=True)
axes[2].set_title('Difference (Unsafe − Safe)')
axes[2].set_xlabel('Next'); axes[2].set_ylabel('Current')

plt.suptitle('Transition Matrices: Safe vs Unsafe Sessions', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '33_markov_safe_vs_unsafe.png', bbox_inches='tight')
plt.show()

# Stationary distributions comparison
pi_safe = stationary_distribution(T_safe)
pi_unsafe = stationary_distribution(T_unsafe)

print('=== Stationary Distributions ===')
print(f'{"State":<12} {"Safe":>10} {"Unsafe":>10} {"Diff":>10}')
print('-' * 45)
for i, state in enumerate(STATE_NAMES):
    print(f'{state:<12} {pi_safe[i]:>10.3f} {pi_unsafe[i]:>10.3f} {pi_unsafe[i]-pi_safe[i]:>+10.3f}')

## 5. Absorbing-State & Drift Analysis

Analyse whether responses in unsafe sessions show progressive drift
by comparing the behavioural state distribution in the **first half**
vs the **second half** of each response.

In [ ]:
# Within-response drift: compare first half vs second half
def compute_half_distributions(sequences, labels):
    """Compare state distributions in first vs second half of responses."""
    results = {'safe': {'first': Counter(), 'second': Counter()},
               'unsafe': {'first': Counter(), 'second': Counter()}}
    
    for seq, is_unsafe in zip(sequences, labels):
        if len(seq) < 4:  # Need at least 4 sentences for meaningful split
            continue
        mid = len(seq) // 2
        label = 'unsafe' if is_unsafe else 'safe'
        
        for s in seq[:mid]:
            results[label]['first'][s] += 1
        for s in seq[mid:]:
            results[label]['second'][s] += 1
    
    return results

half_dists = compute_half_distributions(all_sequences, y)

print('=== Within-Response Drift Analysis ===')
print('Comparing behavioural state proportions: first half vs second half of responses\n')

for label in ['safe', 'unsafe']:
    first_total = sum(half_dists[label]['first'].values())
    second_total = sum(half_dists[label]['second'].values())
    
    print(f'{label.upper()} sessions (first half: {first_total} sentences, second half: {second_total} sentences):')
    print(f'{"State":<12} {"1st Half":>10} {"2nd Half":>10} {"Drift":>10}')
    print('-' * 45)
    
    for state in STATE_NAMES:
        p1 = half_dists[label]['first'].get(state, 0) / max(first_total, 1)
        p2 = half_dists[label]['second'].get(state, 0) / max(second_total, 1)
        print(f'{state:<12} {p1:>10.3f} {p2:>10.3f} {p2-p1:>+10.3f}')
    print()

In [ ]:
# Visualise within-response drift
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, label in enumerate(['safe', 'unsafe']):
    ax = axes[ax_idx]
    first_total = sum(half_dists[label]['first'].values())
    second_total = sum(half_dists[label]['second'].values())
    
    first_props = [half_dists[label]['first'].get(s, 0) / max(first_total, 1) for s in STATE_NAMES]
    second_props = [half_dists[label]['second'].get(s, 0) / max(second_total, 1) for s in STATE_NAMES]
    
    x = np.arange(n_states)
    width = 0.35
    ax.bar(x - width/2, first_props, width, label='First half', color='#3498db', alpha=0.8)
    ax.bar(x + width/2, second_props, width, label='Second half', color='#e74c3c', alpha=0.8)
    
    ax.set_xticks(x)
    ax.set_xticklabels(STATE_NAMES, fontsize=9)
    ax.set_ylabel('Proportion of Sentences')
    ax.set_title(f'{label.title()} Sessions')
    ax.legend(fontsize=9)
    ax.set_ylim(0, 0.6)
    
    # Add drift arrows
    for i in range(n_states):
        drift = second_props[i] - first_props[i]
        if abs(drift) > 0.02:
            color = '#e74c3c' if drift > 0 else '#27ae60'
            ax.annotate(f'{drift:+.2f}', xy=(i, max(first_props[i], second_props[i]) + 0.02),
                       fontsize=8, ha='center', color=color, fontweight='bold')

plt.suptitle('Within-Response Behavioural Drift: First Half vs Second Half', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '34_markov_drift_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
# Mean absorption time: expected sentences to reach each state from each other state
# Using fundamental matrix of Markov chain

def mean_first_passage(T):
    """Compute mean first passage times between all pairs of states."""
    n = T.shape[0]
    pi = stationary_distribution(T)
    
    # Fundamental matrix approach
    Z = np.linalg.inv(np.eye(n) - T + np.outer(np.ones(n), pi))
    M = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                M[i, j] = (Z[j, j] - Z[i, j]) / pi[j]
            else:
                M[i, j] = 1.0 / pi[j]
    return M

M_safe = mean_first_passage(T_safe)
M_unsafe = mean_first_passage(T_unsafe)

print('=== Mean First Passage Times (sentences to reach target state) ===')
print('\nSafe sessions:')
print(pd.DataFrame(M_safe, index=STATE_NAMES, columns=STATE_NAMES).round(1).to_string())
print('\nUnsafe sessions:')
print(pd.DataFrame(M_unsafe, index=STATE_NAMES, columns=STATE_NAMES).round(1).to_string())

# Key comparison: time to reach 'citing' from 'claiming'
t_safe = M_safe[state_to_idx['claiming'], state_to_idx['citing']]
t_unsafe = M_unsafe[state_to_idx['claiming'], state_to_idx['citing']]
print(f'\nMean sentences from claiming → citing:')
print(f'  Safe:   {t_safe:.1f} sentences')
print(f'  Unsafe: {t_unsafe:.1f} sentences')
print(f'  Diff:   {t_unsafe - t_safe:+.1f} (positive = unsafe takes longer to ground claims)')

## 6. Log Findings

In [ ]:
# Build stationary distributions for logging
stationary_log = {}
for cond_col in condition_results:
    stationary_log[cond_col] = {}
    for val, data in condition_results[cond_col].items():
        stationary_log[cond_col][val] = {
            'stationary_dist': {STATE_NAMES[i]: data['stationary_dist'][i]
                               for i in range(n_states)},
            'n_sessions': data['n_sessions'],
            'n_transitions': data['n_transitions']
        }

log_entry = {
    'description': 'Markov chain analysis of sequential behaviour in AI responses',
    'timestamp': datetime.now().isoformat(),
    'seed': SEED,
    'method': {
        'segmentation': 'Sentence-level splitting with regex',
        'states': STATE_NAMES,
        'classification': 'Rule-based keyword matching (citing, hedging, claiming, refusing, neutral)',
        'estimation': 'Maximum likelihood with Laplace smoothing',
    },
    'overall_stats': {
        'total_sentences': total,
        'sentences_per_response': {
            'mean': float(df['n_sentences'].mean()),
            'std': float(df['n_sentences'].std()),
            'min': int(df['n_sentences'].min()),
            'max': int(df['n_sentences'].max()),
        },
        'state_distribution': {s: state_counts[s] / total for s in STATE_NAMES},
    },
    'overall_transition_matrix': T_overall.tolist(),
    'stationary_distributions': stationary_log,
    'safe_vs_unsafe': {
        'safe_stationary': {STATE_NAMES[i]: float(pi_safe[i]) for i in range(n_states)},
        'unsafe_stationary': {STATE_NAMES[i]: float(pi_unsafe[i]) for i in range(n_states)},
        'transition_diff_key': {
            'claiming_to_citing_safe': float(T_safe[state_to_idx['claiming'], state_to_idx['citing']]),
            'claiming_to_citing_unsafe': float(T_unsafe[state_to_idx['claiming'], state_to_idx['citing']]),
        },
        'mean_first_passage_claiming_to_citing': {
            'safe': float(t_safe),
            'unsafe': float(t_unsafe),
        }
    },
}

results_path = RESULTS_DIR / 'markov/metrics.json'
with open(results_path, 'w') as f:
    json.dump(log_entry, f, indent=2, default=str)

print(f'Results saved to: {results_path}')

## Summary

**Key findings:**

- **State transitions reveal response structure:** AI responses follow consistent patterns — typically starting with neutral/contextual content, transitioning through hedging and claiming, and grounding with citations.
- **Unsafe sessions show different transition dynamics:** The transition matrices for safe vs unsafe sessions differ, particularly in the probability of transitioning from claiming to citing (evidence grounding).
- **Scenario drives behavioural state composition** more than pressure or monitoring, consistent with findings from the Bayesian analysis.
- **Within-response drift** shows whether responses progressively shift toward or away from evidence-grounded states.
- **Mean first passage times** quantify how quickly the model returns to evidence-grounded states after making claims — a proxy for "grounding discipline".

**Implications for scalable oversight:**
- Markov chain analysis provides a **structural lens** on AI behaviour beyond binary safe/unsafe classification.
- The transition probability from claiming → citing could serve as a real-time monitoring signal.
- Responses that spend disproportionate time in claiming states without transitioning to citing may flag oversight failures.
- This sequential analysis framework extends naturally to multi-turn conversations where behavioural drift over turns becomes the key safety concern.